In [1]:
import os, json, re, requests
from dotenv import load_dotenv
load_dotenv(override=True)

from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from tavily import TavilyClient
from youtube_search import YoutubeSearch

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

In [2]:
def parse_views(view_str: str) -> int:
    num = re.search(r"[\d,\.]+", view_str)
    if not num:
        return 0
    value = float(num.group().replace(",", ""))
    if "만" in view_str:
        value *= 10000
    elif "억" in view_str:
        value *= 100000000
    return int(value)


@tool
def search_reception_youtube(query: str, max_results: int = 15) -> str:
    """웹소설·웹툰·드라마·영화·뮤지컬이 유튜브에서 어떻게 대중적으로 재생산·소비되는지
    (몰아보기, 편집본, 리뷰 등) 조회수 순으로 검색한다. 결과에는 video_id도 포함되어 있어
    get_youtube_comments 도구와 이어서 쓸 수 있다."""
    results = YoutubeSearch(query, max_results=max_results).to_dict()
    videos = sorted(results, key=lambda r: parse_views(r["views"]), reverse=True)
    lines = []
    for v in videos:
        video_id = v["url_suffix"].split("v=")[-1].split("&")[0]
        lines.append(
            f"[{v['views']}] {v['channel']} | {v['title']} | video_id={video_id}"
        )
    return "\n".join(lines) if lines else "검색 결과가 없습니다."


@tool
def get_youtube_comments(video_id: str, max_results: int = 20) -> str:
    """유튜브 영상 ID를 받아 상위 댓글을 가져온다. search_reception_youtube 결과의
    video_id를 그대로 사용한다. 시청자 반응·수용 양상을 볼 때 사용한다."""
    url = "https://www.googleapis.com/youtube/v3/commentThreads"
    params = {
        "part": "snippet",
        "videoId": video_id,
        "maxResults": max_results,
        "order": "relevance",
        "key": os.environ["YOUTUBE_API_KEY"],
    }
    resp = requests.get(url, params=params)
    if resp.status_code != 200:
        return f"댓글 조회 실패: {resp.status_code} - {resp.json().get('error', {}).get('message', '')}"
    data = resp.json()
    comments = [
        f"[👍{item['snippet']['topLevelComment']['snippet']['likeCount']}] "
        f"{item['snippet']['topLevelComment']['snippet']['textOriginal']}"
        for item in data.get("items", [])
    ]
    return "\n---\n".join(comments) if comments else "댓글이 없거나 댓글 사용이 중지된 영상입니다."


@tool
def search_criticism_web(topic: str) -> str:
    """작품에 대한 평론·비평·분석 기사를 웹에서 검색한다. 여러 각도의 질의로 넓게 검색한 뒤
    위키/블로그/영상 플랫폼을 제외하고 평론 성격이 강한 결과만 걸러 반환한다."""
    queries = [f"{topic} 평론", f"{topic} 비평 분석", f"{topic} 재현 리뷰"]
    exclude_domains = ["namu.wiki", "wikipedia.org", "blog.naver.com", "youtube.com"]
    review_keywords = ["평론", "비평", "리뷰", "분석", "칼럼", "시선", "재현"]

    seen_urls, candidates = set(), []
    for q in queries:
        res = tavily_client.search(
            query=q, max_results=5, search_depth="basic",
            topic="general", include_answer=False, include_raw_content=False,
        )
        for r in res["results"]:
            if r["url"] in seen_urls:
                continue
            seen_urls.add(r["url"])
            if any(d in r["url"] for d in exclude_domains):
                continue
            if any(k in r["title"] for k in review_keywords) or r["score"] > 0.5:
                candidates.append(r)

    candidates.sort(key=lambda x: x["score"], reverse=True)
    lines = [f"[{r['score']:.2f}] {r['title']} | {r['url']}" for r in candidates[:8]]
    return "\n".join(lines) if lines else "평론 성격의 결과를 찾지 못했습니다."


@tool
def extract_article(urls: list[str]) -> str:
    """search_criticism_web에서 찾은 URL들의 본문을 깨끗하게 추출한다.
    실제 평론 내용을 읽고 인용·분석할 때 사용한다."""
    extracted = tavily_client.extract(urls=urls)
    parts = [
        f"URL: {item['url']}\n본문: {item['raw_content'][:1500]}"
        for item in extracted["results"]
    ]
    return "\n\n---\n\n".join(parts) if parts else "본문 추출에 실패했습니다."


tools = [search_reception_youtube, get_youtube_comments, search_criticism_web, extract_article]
tool_map = {t.name: t for t in tools}

MODEL = "google_genai:gemini-flash-latest"
model = init_chat_model(MODEL, temperature=0.2)
model_with_tools = model.bind_tools(tools)

In [3]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode
from langchain_core.messages import SystemMessage, HumanMessage

SYSTEM_PROMPT = SystemMessage(content=(
    "너는 웹소설·웹툰·드라마·영화·뮤지컬 등 대중서사가 어떻게 대중적으로 재생산·소비되고 "
    "(유튜브 조회수·댓글), 동시에 어떻게 비평적으로 논의되는지(평론)를 함께 조사하는 연구 보조원이다. "
    "필요하면 여러 도구를 순차적으로 사용해 충분한 근거를 모은 뒤 종합해서 답하라."
))

def agent_node(state: MessagesState):
    messages = [SYSTEM_PROMPT] + state["messages"]
    response = model_with_tools.invoke(messages)
    return {"messages": [response]}

def should_continue(state: MessagesState):
    last_message = state["messages"][-1]
    return "tools" if last_message.tool_calls else END

graph = StateGraph(MessagesState)
graph.add_node("agent", agent_node)
graph.add_node("tools", ToolNode(tools))
graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
graph.add_edge("tools", "agent")

app = graph.compile()

In [4]:
result = app.invoke({
    "messages": [HumanMessage(content=(
        "옥씨부인전이 유튜브에서 어떻게 소비되는지(조회수 상위 영상과 댓글 반응 포함), "
        "그리고 평론에서는 어떻게 논의되는지 비교해서 정리해줘"
    ))]
}, config={"recursion_limit": 15})

print(result["messages"][-1].content)

[{'type': 'text', 'text': '드라마 **《옥씨부인전》**(JTBC, 임지연·추영우 주연)에 대한 **유튜브 대중적 소비 양상**과 **평론가들의 비평적 논의**를 비교 정리한 내용입니다.\n\n---\n\n### 1. 유튜브에서의 대중적 소비 양상\n\n유튜브에서 《옥씨부인전》은 **‘임지연의 연기 차력쇼’, ‘사이다 신분 상승/복수물’, ‘로맨스&케미스트리’**라는 세 가지 축을 중심으로 활발하게 재생산·소비되고 있습니다.\n\n#### ① 주요 콘텐츠 유형 및 높은 조회수 클립\n* **사이다·카리스마 대립 씬 (조회수 130만 회 이상)**\n  * 과거 자신을 학대했던 전 주인(소혜 등)이나 정체를 의심하는 양반들 앞에서 당황하지 않고 완벽한 양반 ‘옥태영’으로 위장해 압도하는 장면.\n* **결말 및 감동/슬픔 하이라이트 (조회수 100만 회 내외)**\n  * 가짜 서방 성윤겸/천승휘(추영우 분)의 희생과 마지막 약조, 조연들(만석 등)의 비극적이면서도 따뜻한 서사를 모은 클립.\n* **몰아보기 / 숏폼 요약본 (수십만 회)**\n  * *"IQ 200 천재 노비가 최초 여성 변호사로 초고속 신분 상승"*, *"법잘알 도망 노비가 학대 아씨에게 복수하는 법"* 등 **웹소설·사이다 서사 스타일의 타이틀**을 달아 빠르게 소비됨.\n* **로맨스 & 인물 케미 모음 (20만~40만 회)**\n  * 추영우의 \'멍뭉미/대형견 남편\' 모먼트, 임지연과의 애정 씬, 신분을 세탁한 두 노비(임지연x이재원)의 티키타카 및 브로맨스 모음.\n\n#### ② 유튜브 댓글 반응의 특징\n* **임지연 배우의 연기력 찬사 & 《더 글로리》 \'박연진\' 밈(Meme) 활용**\n  * 양반 \'옥태영\'으로 변신해 전 주인을 쏘아붙이거나 비웃을 때, 시청자들은 *"꼭 찾으시길 할 때 박연진 나왔다"*, *"연진아 멋있다, 브라보"*, *"사이다 발성 미쳤다"*며 전작 캐릭터를 연상시키는 쾌감을 즐김.\n  * *"감독이 임지연 캐스팅 안 됐으

In [6]:
from IPython.display import Markdown, display

content = result["messages"][-1].content

if isinstance(content, list):
    text = "\n".join(part["text"] for part in content if isinstance(part, dict) and "text" in part)
else:
    text = content

display(Markdown(text))

드라마 **《옥씨부인전》**(JTBC, 임지연·추영우 주연)에 대한 **유튜브 대중적 소비 양상**과 **평론가들의 비평적 논의**를 비교 정리한 내용입니다.

---

### 1. 유튜브에서의 대중적 소비 양상

유튜브에서 《옥씨부인전》은 **‘임지연의 연기 차력쇼’, ‘사이다 신분 상승/복수물’, ‘로맨스&케미스트리’**라는 세 가지 축을 중심으로 활발하게 재생산·소비되고 있습니다.

#### ① 주요 콘텐츠 유형 및 높은 조회수 클립
* **사이다·카리스마 대립 씬 (조회수 130만 회 이상)**
  * 과거 자신을 학대했던 전 주인(소혜 등)이나 정체를 의심하는 양반들 앞에서 당황하지 않고 완벽한 양반 ‘옥태영’으로 위장해 압도하는 장면.
* **결말 및 감동/슬픔 하이라이트 (조회수 100만 회 내외)**
  * 가짜 서방 성윤겸/천승휘(추영우 분)의 희생과 마지막 약조, 조연들(만석 등)의 비극적이면서도 따뜻한 서사를 모은 클립.
* **몰아보기 / 숏폼 요약본 (수십만 회)**
  * *"IQ 200 천재 노비가 최초 여성 변호사로 초고속 신분 상승"*, *"법잘알 도망 노비가 학대 아씨에게 복수하는 법"* 등 **웹소설·사이다 서사 스타일의 타이틀**을 달아 빠르게 소비됨.
* **로맨스 & 인물 케미 모음 (20만~40만 회)**
  * 추영우의 '멍뭉미/대형견 남편' 모먼트, 임지연과의 애정 씬, 신분을 세탁한 두 노비(임지연x이재원)의 티키타카 및 브로맨스 모음.

#### ② 유튜브 댓글 반응의 특징
* **임지연 배우의 연기력 찬사 & 《더 글로리》 '박연진' 밈(Meme) 활용**
  * 양반 '옥태영'으로 변신해 전 주인을 쏘아붙이거나 비웃을 때, 시청자들은 *"꼭 찾으시길 할 때 박연진 나왔다"*, *"연진아 멋있다, 브라보"*, *"사이다 발성 미쳤다"*며 전작 캐릭터를 연상시키는 쾌감을 즐김.
  * *"감독이 임지연 캐스팅 안 됐으면 드라마 엎었을 거라 했는데 진짜 원톱으로 화면을 장악했다"*는 연기력 극찬이 주를 이룸.
* **비극적 인물들에 대한 감정적 여운과 눈물**
  * 주인공을 위해 희생한 성윤겸(추영우) 및 노비 친구들의 서사에 *"옷 바꿔 입어 살려준 은혜를 옷 바꿔 갚았다"*, *"너무 슬프고 아련하다"*며 깊은 감정적 몰입을 보임.

---

### 2. 평론 및 비평적 논의

평론가와 대중문화 비평 기사에서는 《옥씨부인전》을 **퓨전사극의 장르적 진화, 신분제와 현대적 갑질에 대한 사회적 메시지, 그리고 신분 교체 서사의 주체적 변주** 관점에서 논의하고 있습니다.

#### ① 퓨전사극 장르의 진화 ("퓨전사극의 퓨처")
* 기존 퓨전사극이 주로 가벼운 로맨틱 코미디에 그쳤던 것과 달리, 조선 시대 대리 소송인인 **‘외지부(조선시대 변호사)’**라는 소재를 적극 활용해 **법정 드라마·휴먼 서사·정치극을 유기적으로 결합**했다고 평가합니다.
* 답답한 사건(고구마)과 지혜로운 해결(사이다)의 완급 조절이 뛰어났다는 평을 받습니다.

#### ② 신분제 폭력성과 현대적 시의성
*主人公의 본래 이름인 **'구덕이(구더기)'**라는 비인간적 명칭과, 노비의 목숨값이 말 한 마리 값(서른 냥)으로 치부되던 조선 사회의 참혹한 계급 구조를 직시합니다.
* 비평가들은 드라마 속 신분적 폭력과 갑질이 단지 과거의 이야기가 아니라, **현대 사회의 불평등, 갑질, 법적 사각지대**와 겹쳐 읽히며 *"사극이지만 지금 우리의 이야기"*라는 메시지를 던진다고 분석합니다.

#### ③ 신분 교체 서사의 변주와 '롤모델'의 정신 계승
* 일반적인 '왕자와 거지' 식의 신분 교체물과 달리, 원래 아씨였던 **진짜 옥태영(손나은 분)**은 구덕이가 부정하거나 뛰어넘어야 할 대상이 아닙니다.
* 진짜 옥태영은 구덕이의 **정신적 롤모델이자 이상향**으로 작동하며, 구덕이는 진짜 옥태영의 신분뿐만 아니라 **'약자를 돕고자 했던 뜻'을 주체적으로 계승**하여 여성 히어로이자 법률가로 성장합니다.

---

### 3. 비교 종합 (유튜브 대중 수용 vs 비평적 논의)

| 구분 | **유튜브 대중 수용 (재생산·소비)** | **평론 및 비평적 논의** |
| :--- | :--- | :--- |
| **초점** | **배우 연기 / 직관적 쾌감 / 서사적 감동** | **장르적 성취 / 사회적 메시지 / 서사 구조 분석** |
| **주요 화두** | • 임지연의 카리스마 연기 및 '박연진' 모먼트<br>• 남주(추영우)와의 로맨스 & 멍뭉미 케미<br>• 학대 주인을 응징하는 사이다 쾌감과 희생에 대한 눈물 | • 조선시대 '외지부' 소재를 통한 퓨전사극의 확장<br>• 계급 사회와 현대 '갑질' 불평등에 대한 시의성<br>• 진짜 '옥태영'의 정신을 계승하는 주체적 여성상 |
| **소비 방식** | 하이라이트, 숏폼, 짤, '사이다 복수물' 형태의 빠른 소비 | 작품의 역사적 배경, 신분 교체 모티브의 변주, 메시지 분석 |

* **공통점**: 대중과 평론가 모두 **주인공 임지연의 완벽한 캐릭터 소화력**과, 억울한 자들을 돕는 **'외지부'로서의 주체적이고 통쾌한 활약**을 작품의 최고 강점으로 꼽았습니다.